# BanglaT5 QA — Full Fine-Tuning
*Full parameter fine-tuning of BanglaT5-small on BanglaRQA*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

---

💡 **Hardware Note**: A GPU runtime is strongly recommended for fine-tuning.
In Google Colab, select: **Runtime > Change runtime type > Hardware accelerator > GPU (T4)**.

---


## 1. Setup & Dependencies

Install required libraries for Hugging Face Transformers, Datasets, PEFT, and CodeCarbon emissions tracking.


In [1]:
# Uncomment and run if running on Google Colab:
# !pip install -q transformers>=4.40 datasets sentencepiece accelerate peft codecarbon git+https://github.com/csebuetnlp/normalizer

import sys
import subprocess

# Ensure essential dependencies are installed
required_packages = ["transformers", "datasets", "sentencepiece", "accelerate", "codecarbon", "normalizer"]
if False:
    required_packages.append("peft")

for pkg in required_packages:
    try:
        __import__(pkg)
    except ImportError:
        print(f"Installing {pkg}...")
        install_target = "git+https://github.com/csebuetnlp/normalizer" if pkg == "normalizer" else pkg
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", install_target])

print("All dependencies verified successfully.")


Installing codecarbon...
Installing normalizer...
All dependencies verified successfully.


## 2. Environment & Project Directories

Locate the project folder (works seamlessly when running directly in Colab, from Google Drive, or locally).


In [4]:
import os
import torch
from pathlib import Path

# Verify GPU accelerator
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Device: {device}")
if torch.cuda.is_available():
    print(f"GPU Model:      {torch.cuda.get_device_name(0)}")
    print(f"VRAM Available: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

# Automatically detect your Google Drive folder or Colab folder
potential_roots = [
    Path("/content/drive/MyDrive/jakirJisan"),
    Path("/content/drive/MyDrive/BanglaT5_Project"),
    Path("/content/jakirJisan"),
    Path("/content/BanglaT5_Project"),
    Path(".").resolve()
]

PROJECT_ROOT = None
for p in potential_roots:
    if p.exists() and (p / "data").exists():
        PROJECT_ROOT = p
        break

if PROJECT_ROOT is None:
    for p in potential_roots:
        if p.exists():
            PROJECT_ROOT = p
            break

if PROJECT_ROOT is None:
    PROJECT_ROOT = Path(".").resolve()

# Setup subdirectories
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PREPARED = PROJECT_ROOT / "data" / "prepared"
MODELS_DIR = PROJECT_ROOT / "models" / "full_ft"
RESULTS_DIR = PROJECT_ROOT / "results"
CARBON_DIR = PROJECT_ROOT / "carbon_logs" / "full_ft"

for directory in [DATA_RAW, DATA_PREPARED, MODELS_DIR, RESULTS_DIR, CARBON_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Project Root: {PROJECT_ROOT}")
print(f"Raw Data Dir: {DATA_RAW}")


Active Device: cuda
GPU Model:      Tesla T4
VRAM Available: 14.56 GB
Project Root: /content
Raw Data Dir: /content/data/raw


## 3. Data Loading & Prompt Formatting

Load the domain-targeted BanglaRQA splits. The dataset is formatted into question-answering prompts:
```text
প্রসঙ্গ: <Context Passage>
প্রশ্ন: <Question Text>
উত্তর:
```


In [5]:
import ast
import re
import shutil
import unicodedata
import subprocess
import pandas as pd
from datasets import Dataset, load_from_disk
from normalizer import normalize

MODEL_NAME = "csebuetnlp/banglat5_small"
MAX_SOURCE_LENGTH = 768
MAX_TARGET_LENGTH = 96
NO_ANSWER_TAG = "উত্তর নেই"

# 1. Automatically ensure the 3 raw CSV dataset splits exist in DATA_RAW
raw_files = {
    "BanglaRQA_targeted_domains.csv": "18w1wty89i9osl51ad1iQAHkxobP89x0B",
    "validation_targeted.csv": "1JTkFEIcwXkutDjhXdbWfHazv7Zz4FZdE",
    "test_targeted.csv": "1XkdTDdw4_IshTLMnQuprFJcJVhPTftaX"
}

for fname, fid in raw_files.items():
    target_path = DATA_RAW / fname
    if not target_path.exists():
        # If downloaded previously to /data/raw, copy it over!
        if (Path("/data/raw") / fname).exists():
            shutil.copy(Path("/data/raw") / fname, target_path)
            print(f"Copied {fname} from /data/raw to {target_path}")
        else:
            print(f"Downloading {fname} directly to {target_path}...")
            subprocess.run(["pip", "install", "-q", "gdown"], check=False)
            import gdown
            gdown.download(id=fid, output=str(target_path), quiet=False)
    else:
        print(f"Found: {fname}")

def clean_bangla(text):
    if pd.isna(text) or not text:
        return ""
    text = unicodedata.normalize("NFC", str(text))
    text = normalize(text)
    return re.sub(r"\s+", " ", text).strip()

def parse_answers(df):
    def extract_answers(val):
        if pd.isna(val):
            return []
        match = re.search(r"'answer_text'\s*:\s*array\((\[.*?\])\s*,\s*dtype=object\)", str(val), re.DOTALL)
        if match:
            try:
                res = ast.literal_eval(match.group(1))
                return [clean_bangla(a) for a in res if clean_bangla(a)]
            except:
                pass
        return [clean_bangla(x) for x in str(val).split("|") if clean_bangla(x)]

    df = df.copy()
    col = "answers" if "answers" in df.columns else "answer_text"
    df["reference_answers"] = df.apply(
        lambda row: [NO_ANSWER_TAG] if int(row.get("is_answerable", 1)) == 0 else extract_answers(row.get(col)),
        axis=1
    )
    df["target_text"] = df["reference_answers"].apply(lambda refs: refs[0] if refs else NO_ANSWER_TAG)
    return df

# 2. Load prepared datasets if available, otherwise process raw files
train_parquet = DATA_PREPARED / "train_processed.parquet"
if train_parquet.exists():
    print("Loading preprocessed dataset from data/prepared/...")
    processed_train = pd.read_parquet(DATA_PREPARED / "train_processed.parquet")
    processed_val = pd.read_parquet(DATA_PREPARED / "validation_processed.parquet")
    processed_test = pd.read_parquet(DATA_PREPARED / "test_processed.parquet")
else:
    print("Processing raw CSV files from data/raw/...")
    raw_train = pd.read_csv(DATA_RAW / "BanglaRQA_targeted_domains.csv")
    raw_val = pd.read_csv(DATA_RAW / "validation_targeted.csv")
    raw_test = pd.read_csv(DATA_RAW / "test_targeted.csv")

    for df in [raw_train, raw_val, raw_test]:
        df["context_clean"] = df["context"].apply(clean_bangla)
        df["question_clean"] = df["question_text"].apply(clean_bangla)
        df["input_text"] = "প্রসঙ্গ: " + df["context_clean"] + "\nপ্রশ্ন: " + df["question_clean"] + "\nউত্তর:"

    processed_train = parse_answers(raw_train)
    processed_val = parse_answers(raw_val)
    processed_test = parse_answers(raw_test)

print(f"Loaded splits: Train={len(processed_train)}, Validation={len(processed_val)}, Test={len(processed_test)}")


Downloading...
From: https://drive.google.com/uc?id=18w1wty89i9osl51ad1iQAHkxobP89x0B
To: /content/data/raw/BanglaRQA_targeted_domains.csv
100%|██████████| 17.1M/17.1M [00:00<00:00, 172MB/s]


Downloading...
From: https://drive.google.com/uc?id=1JTkFEIcwXkutDjhXdbWfHazv7Zz4FZdE
To: /content/data/raw/validation_targeted.csv
100%|██████████| 4.23M/4.23M [00:00<00:00, 161MB/s]


Downloading...
From: https://drive.google.com/uc?id=1XkdTDdw4_IshTLMnQuprFJcJVhPTftaX
To: /content/data/raw/test_targeted.csv
100%|██████████| 4.62M/4.62M [00:00<00:00, 141MB/s]


Processing raw CSV files from data/raw/...
Loaded splits: Train=3035, Validation=831, Test=886


In [6]:
# Inspect a formatted prompt sample
sample_row = processed_train.iloc[0]
print("=== SAMPLE INPUT PROMPT ===")
print(sample_row["input_text"][:400] + "...\n")
print("=== SAMPLE TARGET ANSWER ===")
print(sample_row["target_text"])
print("=== ALL ACCEPTED REFERENCES ===")
print(sample_row["reference_answers"])


=== SAMPLE INPUT PROMPT ===
প্রসঙ্গ: যেহেতু ম্যাকওএস পিওএসআইএক্স সমর্থিত, তাই লিনাক্স সহ অন্যান্য ইউনিক্স-সদৃশ সিস্টেমের জন্য লেখা অনেক সফ্টওয়্যার প্যাকেজ অনেক বৈজ্ঞানিক এবং প্রযুক্তিগত সফ্টওয়্যার সহ এটিতে চালানোর জন্য পুনরায় কম্পাইল করা যায়। থার্ড-পার্টি প্রোজেক্ট যেমন হোমব্রু, ফিঙ্ক, ম্যাকপোর্টস, এবং পূর্ব-সংকলিত বা প্রাক-বিন্যাসিত প্যাকেজ প্রদান করে। অ্যাপল এবং অন্যরা এক্স উইন্ডো সিস্টেম গ্রাফিকাল ইন্টারফেসের সংস্করণ ...

=== SAMPLE TARGET ANSWER ===
হ্যাঁ
=== ALL ACCEPTED REFERENCES ===
['হ্যাঁ', 'হ্যাঁ']


## 4. Tokenizer & Dataset Tokenization

Initialize the SentencePiece tokenizer for `csebuetnlp/banglat5_small` and convert texts to input IDs and attention masks.


In [7]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

def tokenize_function(examples):
    model_inputs = tokenizer(
        examples["input_text"],
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
        padding=False
    )
    labels = tokenizer(
        text_target=examples["target_text"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding=False
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Check for existing tokenized disk cache or build dataset
tok_train_dir = DATA_PREPARED / "tokenized_train"
if tok_train_dir.exists():
    print("Loading cached tokenized datasets from disk...")
    tokenized_train = load_from_disk(str(DATA_PREPARED / "tokenized_train"))
    tokenized_val = load_from_disk(str(DATA_PREPARED / "tokenized_validation"))
    tokenized_test = load_from_disk(str(DATA_PREPARED / "tokenized_test"))
else:
    print("Tokenizing datasets on the fly...")
    ds_train = Dataset.from_pandas(processed_train)
    ds_val = Dataset.from_pandas(processed_val)
    ds_test = Dataset.from_pandas(processed_test)

    tokenized_train = ds_train.map(tokenize_function, batched=True, remove_columns=ds_train.column_names)
    tokenized_val = ds_val.map(tokenize_function, batched=True, remove_columns=ds_val.column_names)
    tokenized_test = ds_test.map(tokenize_function, batched=True, remove_columns=ds_test.column_names)

print(f"Tokenized dataset ready: Train={len(tokenized_train)}, Val={len(tokenized_val)}, Test={len(tokenized_test)}")


config.json:   0%|          | 0.00/646 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.83k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B / 1.11MB            

spiece.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/1.79k [00:00<?, ?B/s]

Tokenizing datasets on the fly...


Map:   0%|          | 0/3035 [00:00<?, ? examples/s]

Map:   0%|          | 0/831 [00:00<?, ? examples/s]

Map:   0%|          | 0/886 [00:00<?, ? examples/s]

Tokenized dataset ready: Train=3035, Val=831, Test=886


## 5. Model Architecture & Full Fine-Tuning Setup

Load [`csebuetnlp/banglat5_small`](https://huggingface.co/csebuetnlp/banglat5_small) encoder-decoder model.


In [8]:
import gc
from transformers import AutoModelForSeq2SeqLM

# Clean GPU memory before model allocation
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Load base sequence-to-sequence model for full fine-tuning
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("=" * 60)
print("Full Fine-Tuning Setup:")
print(f"Total Parameters:     {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,} (100.0%)")
print("=" * 60)


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  242MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

Full Fine-Tuning Setup:
Total Parameters:     60,506,624
Trainable Parameters: 60,506,624 (100.0%)


## 6. Training Configuration & CodeCarbon Tracking

Configure training hyperparameters and attach CodeCarbon's `EmissionsTracker` to measure energy consumption and environmental impact.


In [9]:
import time
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
from codecarbon import EmissionsTracker

output_dir = MODELS_DIR / "trainer_output"

# Define Hugging Face Seq2Seq training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir=str(output_dir),
    num_train_epochs=3,
    learning_rate=1e-3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,   # Effective batch size = 2 x 4 = 8
    max_grad_norm=1.0,
    fp16=False,                      # FP32 precision
    seed=42,
    data_seed=42,
    logging_steps=50,
    eval_strategy="no",
    save_strategy="no",
    predict_with_generate=False,
    report_to="none"
)

# Dynamic collator with padding
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt"
)

# Initialize Seq2Seq Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator
)


## 7. Model Training

Train the model for 3 epochs while actively recording power draw and carbon emissions.


In [10]:
# Initialize emissions tracker
emissions_file = CARBON_DIR / "emissions.csv"
if emissions_file.exists():
    emissions_file.unlink()

tracker = EmissionsTracker(
    project_name="BanglaT5_full_ft",
    output_dir=str(CARBON_DIR),
    output_file="emissions.csv",
    measure_power_secs=5,
    save_to_file=True,
    save_to_api=False,
    log_level="error"
)

# Run training
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

print("Starting training with CodeCarbon emissions tracking...")
tracker.start()
start_time = time.perf_counter()

train_result = trainer.train()

end_time = time.perf_counter()
tracker.stop()
train_time_sec = end_time - start_time

peak_vram_gb = torch.cuda.max_memory_allocated() / (1024**3) if torch.cuda.is_available() else 0.0

# Read emissions metrics
emissions_df = pd.read_csv(emissions_file)
energy_kwh = float(emissions_df.iloc[-1]["energy_consumed"])
co2_kg = float(emissions_df.iloc[-1]["emissions"])

print("\n" + "=" * 60)
print("TRAINING FINISHED")
print(f"Total Time:      {train_time_sec:.2f} s ({train_time_sec / 60:.2f} min)")
print(f"Peak GPU VRAM:   {peak_vram_gb:.3f} GB")
print(f"Energy Consumed: {energy_kwh:.6f} kWh")
print(f"CO2 Emissions:   {co2_kg:.8f} kg CO2eq")
print("=" * 60)


[codecarbon WARNING @ 14:51:46] Multiple instances of codecarbon are allowed to run at the same time.


Starting training with CodeCarbon emissions tracking...


Step,Training Loss
50,20.830942
100,10.911603
150,9.834487
200,8.497297
250,7.435779
300,8.200117
350,7.556882
400,6.510413
450,6.763593
500,6.386059



TRAINING FINISHED
Total Time:      460.31 s (7.67 min)
Peak GPU VRAM:   1.978 GB
Energy Consumed: 0.009965 kWh
CO2 Emissions:   0.00469146 kg CO2eq


## 8. Evaluation — Exact Match (EM) & Token-level F1

Evaluate using greedy decoding (`num_beams=1`, `max_new_tokens=96`) on both Validation and Test sets.
Predictions and reference answers are normalized (Bengali normalizer + punctuation removal) before scoring.


In [11]:
import numpy as np
from collections import Counter

def normalize_eval_text(text):
    if not text:
        return ""
    text = clean_bangla(text).lower()
    return "".join(c for c in text if not unicodedata.category(c).startswith("P")).strip()

def compute_exact_match(pred, ref):
    return int(normalize_eval_text(pred) == normalize_eval_text(ref))

def compute_token_f1(pred, ref):
    p_tokens = normalize_eval_text(pred).split()
    r_tokens = normalize_eval_text(ref).split()
    if not p_tokens and not r_tokens:
        return 1.0
    if not p_tokens or not r_tokens:
        return 0.0
    overlap = sum((Counter(p_tokens) & Counter(r_tokens)).values())
    if overlap == 0:
        return 0.0
    prec = overlap / len(p_tokens)
    rec = overlap / len(r_tokens)
    return 2 * prec * rec / (prec + rec)

def evaluate_dataset(split_name, tokenized_ds, original_df):
    trainer.args.predict_with_generate = True
    raw_preds = trainer.predict(
        tokenized_ds,
        num_beams=1,
        do_sample=False,
        max_new_tokens=MAX_TARGET_LENGTH
    )
    pred_tokens = raw_preds.predictions[0] if isinstance(raw_preds.predictions, tuple) else raw_preds.predictions
    pred_tokens = np.where(pred_tokens == -100, tokenizer.pad_token_id, pred_tokens)
    decoded_preds = tokenizer.batch_decode(pred_tokens, skip_special_tokens=True)

    results_df = original_df.copy()
    results_df["prediction"] = decoded_preds

    # Score against all accepted reference answers
    scores = results_df.apply(
        lambda r: (
            max(compute_exact_match(r["prediction"], ref) for ref in r["reference_answers"]),
            max(compute_token_f1(r["prediction"], ref) for ref in r["reference_answers"])
        ),
        axis=1
    )
    results_df["EM"] = [s[0] for s in scores]
    results_df["F1"] = [s[1] for s in scores]

    em_score = 100 * results_df["EM"].mean()
    f1_score = 100 * results_df["F1"].mean()
    blanks = (results_df["prediction"].str.strip() == "").sum()

    print(f"{split_name.upper():12s} | EM: {em_score:6.2f}% | F1: {f1_score:6.2f}% | Blanks: {blanks}/{len(results_df)}")
    return results_df, em_score, f1_score, blanks

print("Evaluating model performance...")
val_results_df, val_em, val_f1, val_blanks = evaluate_dataset("Validation", tokenized_val, processed_val)
test_results_df, test_em, test_f1, test_blanks = evaluate_dataset("Test", tokenized_test, processed_test)


Evaluating model performance...


[transformers] Both `max_new_tokens` (=96) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=96) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=96) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=96) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs

VALIDATION   | EM:  48.50% | F1:  63.59% | Blanks: 0/831


[transformers] Both `max_new_tokens` (=96) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=96) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=96) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=96) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs

TEST         | EM:  47.97% | F1:  61.98% | Blanks: 0/886


## 9. Save Model & Summary Results

Persist the trained weights, tokenizer snapshot, and performance summary table.


In [12]:
final_model_dir = MODELS_DIR / "final_model"
final_model_dir.mkdir(parents=True, exist_ok=True)

# Save weights and tokenizer
trainer.save_model(str(final_model_dir))
tokenizer.save_pretrained(str(final_model_dir))

# Build summary record
summary_data = {
    "method": "Full Fine-Tuning",
    "rank": None,
    "total_parameters": total_params,
    "trainable_parameters": trainable_params,
    "trainable_percent": 100 * trainable_params / total_params,
    "validation_em": val_em,
    "validation_f1": val_f1,
    "validation_blank_percent": 100 * val_blanks / len(processed_val),
    "test_em": test_em,
    "test_f1": test_f1,
    "test_blank_percent": 100 * test_blanks / len(processed_test),
    "training_time_seconds": train_time_sec,
    "peak_gpu_memory_gb": peak_vram_gb,
    "energy_kwh": energy_kwh,
    "co2_kg": co2_kg
}

summary_df = pd.DataFrame([summary_data])
summary_df.to_csv(RESULTS_DIR / "full_ft_summary.csv", index=False)
test_results_df.to_csv(RESULTS_DIR / "full_ft_test_predictions.csv", index=False)
val_results_df.to_csv(RESULTS_DIR / "full_ft_validation_predictions.csv", index=False)

print(f"Model saved to:   {final_model_dir}")
print(f"Results saved to: {RESULTS_DIR / 'full_ft_summary.csv'}")
summary_df


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to:   /content/models/full_ft/final_model
Results saved to: /content/results/full_ft_summary.csv


,method,rank,total_parameters,trainable_parameters,trainable_percent,validation_em,validation_f1,validation_blank_percent,test_em,test_f1,test_blank_percent,training_time_seconds,peak_gpu_memory_gb,energy_kwh,co2_kg
0,Full Fine-Tuning,None,60506624,60506624,100.0,48.495788,63.592698,0.0,47.968397,61.976864,0.0,460.313661,1.977525,0.009965,0.004691


## 10. Interactive Sample Predictions

Demonstrate the model on real test questions from the BanglaRQA dataset.


In [13]:
# Display qualitative test predictions
samples = test_results_df.sample(5, random_state=42)

for i, (_, row) in enumerate(samples.iterrows(), 1):
    print("=" * 70)
    print(f"SAMPLE #{i}")
    print("=" * 70)
    print("QUESTION:   ", row["question_clean"])
    print("REFERENCE:  ", row["reference_answers"])
    print("PREDICTION: ", row["prediction"])
    print(f"SCORES:      EM={row['EM']} | F1={row['F1']:.2f}")
    print()


SAMPLE #1
QUESTION:    জন উইক এ কাকে কাকে অভিনয় করতে দেখা যায়?
REFERENCE:   ['কিয়ানু রিভস; মাইকেল নাইকভিস্ট; আলফি এলেন; আদ্রিয়ান্নে পালিস্কি; ব্রিডজেট ময়নাহান; ডিন উইন্টারস; ইয়ান ম্যাকশেন; জন লেগুইযামো; উইলেম ডাফো', 'কিয়ানু রিভস; মাইকেল নাইকভিস্ট; আলফি এলেন; আদ্রিয়ান্নে পালিস্কি; ব্রিডজেট ময়নাহান; ডিন উইন্টারস; ইয়ান ম্যাকশেন; জন লেগুইযামো; উইলেম ডাফো']
PREDICTION:  কিয়ানু রিভস; মাইকেল নাইকভিস্ট; আলফি এলেন; ডিন উইন্টারস; ইয়ান ম্যাকশেন; উইলেম ডাফো
SCORES:      EM=0 | F1=0.80

SAMPLE #2
QUESTION:    লিনাক্স মিন্ট কী ভিত্তিক লিনাক্স সম্প্রদায়-চালিত একটি গ্নু/লিনাক্স ডিস্ট্রিবিউশন?
REFERENCE:   ['উবুন্টু-ভিত্তিক', 'উবুন্টু-ভিত্তিক']
PREDICTION:  ডেবিয়ান; উবুন্টু
SCORES:      EM=0 | F1=0.00

SAMPLE #3
QUESTION:    এরশাদ কত সালে আনুষ্ঠানিকভাবে রাষ্ট্রপতি পদে অধিষ্ঠিত হন?
REFERENCE:   ['১৯৮৩', '১৯৮৩']
PREDICTION:  ১৯৮৩
SCORES:      EM=1 | F1=1.00

SAMPLE #4
QUESTION:    জীবন বীমা কাদের মধ্যে সম্পাদিত হয়?
REFERENCE:   ['বীমা গ্রহীতা ও একটি বীমা প্রতিষ্ঠানের', 'কজন বীমা গ্রহীতা ও 